# Test Set Evaluation: Firescar Segmentation

Evaluate the best model checkpoint on the held-out test set. Reports IoU, F1, precision, recall with per-threshold analysis and confusion matrix.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.abspath(".."))
from firescars.dataset import FirescarDataset
from firescars.evaluate import compute_metrics
from firescars.model import FirescarModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_PATH = "/tmp/firescar_chips"
CKPT_PATH = "/tmp/firescar_checkpoints/model_best.pth"
print(f"Device: {device}")

## 1. Load Best Model

In [ ]:
model = FirescarModel(
    encoder_name="vit_base_patch16_224",
    in_chans=6,
    img_size=224,
    pretrained_encoder=False,
).to(device)

ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.eval()

print(f"Loaded checkpoint from epoch {ckpt['epoch'] + 1}")
print(f"Best val_loss: {ckpt['best_val_loss']:.4f}")

## 2. Run Inference on Test Set

In [ ]:
test_ds = FirescarDataset(DATA_PATH, split="test", augment=False)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

all_logits = []
all_targets = []

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        all_logits.append(logits.cpu())
        all_targets.append(masks)

all_logits = torch.cat(all_logits)
all_targets = torch.cat(all_targets)
all_probs = torch.sigmoid(all_logits)

print(f"Test samples: {len(all_logits)}")
print(f"Predictions shape: {all_probs.shape}")

## 3. Metrics at Default Threshold (0.5)

In [ ]:
metrics = compute_metrics(all_logits, all_targets, threshold=0.5)

print("=" * 50)
print("TEST SET RESULTS (threshold=0.5)")
print("=" * 50)
print(f"  IoU:       {metrics['iou']:.4f}  (target: >= 0.75)")
print(f"  F1:        {metrics['f1']:.4f}  (target: >= 0.80)")
print(f"  Precision: {metrics['precision']:.4f}  (target: >= 0.80)")
print(f"  Recall:    {metrics['recall']:.4f}  (target: >= 0.75)")
print("=" * 50)

## 4. Threshold Sweep (Precision-Recall Curve)

In [ ]:
thresholds = np.arange(0.1, 0.95, 0.05)
sweep_metrics = {"threshold": [], "iou": [], "f1": [], "precision": [], "recall": []}

for t in thresholds:
    m = compute_metrics(all_logits, all_targets, threshold=t)
    sweep_metrics["threshold"].append(t)
    for k in ["iou", "f1", "precision", "recall"]:
        sweep_metrics[k].append(m[k])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# IoU and F1 vs threshold
axes[0].plot(sweep_metrics["threshold"], sweep_metrics["iou"], "g-o", markersize=5, label="IoU")
axes[0].plot(sweep_metrics["threshold"], sweep_metrics["f1"], "m-o", markersize=5, label="F1")
axes[0].axhline(y=0.75, color="g", linestyle="--", alpha=0.5)
axes[0].axhline(y=0.80, color="m", linestyle="--", alpha=0.5)
axes[0].axvline(x=0.5, color="k", linestyle=":", alpha=0.5, label="Default (0.5)")
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Score")
axes[0].set_title("IoU & F1 vs Threshold")
axes[0].legend()
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3)

# Precision-Recall curve
axes[1].plot(sweep_metrics["recall"], sweep_metrics["precision"], "b-o", markersize=5)
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)
# Annotate best F1 point
best_f1_idx = np.argmax(sweep_metrics["f1"])
axes[1].scatter(
    sweep_metrics["recall"][best_f1_idx],
    sweep_metrics["precision"][best_f1_idx],
    s=100,
    c="red",
    zorder=5,
    label=f"Best F1={sweep_metrics['f1'][best_f1_idx]:.3f} @ t={sweep_metrics['threshold'][best_f1_idx]:.2f}",
)
axes[1].legend()

plt.tight_layout()
plt.savefig("test_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Confusion Matrix

In [ ]:
preds = (all_probs > 0.5).numpy().flatten()
targets = all_targets.numpy().flatten()

tp = ((preds == 1) & (targets == 1)).sum()
fp = ((preds == 1) & (targets == 0)).sum()
fn = ((preds == 0) & (targets == 1)).sum()
tn = ((preds == 0) & (targets == 0)).sum()

cm = np.array([[tn, fp], [fn, tp]])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Unburned", "Burned"])
ax.set_yticklabels(["Unburned", "Burned"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix (Test Set)")

# Annotate cells
for i in range(2):
    for j in range(2):
        val = cm[i, j]
        pct = val / cm.sum() * 100
        ax.text(
            j,
            i,
            f"{val:,}\n({pct:.1f}%)",
            ha="center",
            va="center",
            fontsize=12,
            color="white" if val > cm.max() / 2 else "black",
        )

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Per-Sample IoU Distribution

In [ ]:
per_sample_iou = []
for i in range(len(all_logits)):
    m = compute_metrics(all_logits[i : i + 1], all_targets[i : i + 1])
    per_sample_iou.append(m["iou"])

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(per_sample_iou, bins=20, color="steelblue", edgecolor="white", alpha=0.8)
ax.axvline(
    x=np.mean(per_sample_iou),
    color="red",
    linestyle="--",
    label=f"Mean IoU: {np.mean(per_sample_iou):.3f}",
)
ax.axvline(x=0.75, color="green", linestyle="--", alpha=0.7, label="Target: 0.75")
ax.set_xlabel("IoU")
ax.set_ylabel("Count")
ax.set_title("Per-Sample IoU Distribution (Test Set)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("iou_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(
    f"Samples above target (IoU >= 0.75): {sum(1 for x in per_sample_iou if x >= 0.75)}/{len(per_sample_iou)}"
)